# 01 · La table maître : une ligne par Pokémon

Le but de ce notebook c'est de construire **la table de base du projet** : une ligne = une espèce de Pokémon (repérée par son numéro de Pokédex), avec tous ses attributs (types, stats, taille, poids, génération, stade d'évolution, légendaire, starter…) et les mesures de popularité qui ont déjà un numéro de Pokédex.

Toutes les autres analyses (sur/sous-cotés, conception, réseau, noms) partent de cette table.

On commence par découvrir chaque fichier de PokéAPI avant de les joindre.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 50)
RAW = Path("../data/raw")
OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

## 1. Les espèces (`pokemon_species.csv`)

C'est le fichier central : une ligne par espèce, avec la génération, l'espèce dont elle évolue, la couleur officielle, la forme du corps et les drapeaux légendaire / fabuleux / bébé.

In [2]:
species = pd.read_csv(RAW / "pokeapi/pokemon_species.csv")
print(species.shape)
species.head()

(1025, 20)


,id,identifier,generation_id,evolves_from_species_id,evolution_chain_id,color_id,shape_id,habitat_id,gender_rate,capture_rate,base_happiness,is_baby,hatch_counter,has_gender_differences,growth_rate_id,forms_switchable,is_legendary,is_mythical,order,conquest_order
0,1,bulbasaur,1,NaN,1,5,8,3.0,1,45,70,0,20,0,4,0,0,0,1,NaN
1,2,ivysaur,1,1.0,1,5,8,3.0,1,45,70,0,20,0,4,0,0,0,2,NaN
2,3,venusaur,1,2.0,1,5,8,3.0,1,45,70,0,20,1,4,1,0,0,3,NaN
3,4,charmander,1,NaN,2,8,6,4.0,1,45,70,0,20,0,4,0,0,0,4,109.0
4,5,charmeleon,1,4.0,2,8,6,4.0,1,45,70,0,20,0,4,0,0,0,5,110.0


In [3]:
species.dtypes

id                           int64
identifier                     str
generation_id                int64
evolves_from_species_id    float64
evolution_chain_id           int64
color_id                     int64
shape_id                     int64
habitat_id                 float64
gender_rate                  int64
capture_rate                 int64
base_happiness               int64
is_baby                      int64
hatch_counter                int64
has_gender_differences       int64
growth_rate_id               int64
forms_switchable             int64
is_legendary                 int64
is_mythical                  int64
order                        int64
conquest_order             float64
dtype: object

In [4]:
species.isna().sum()

id                           0
identifier                   0
generation_id                0
evolves_from_species_id    541
evolution_chain_id           0
color_id                     0
shape_id                     0
habitat_id                 639
gender_rate                  0
capture_rate                 0
base_happiness               0
is_baby                      0
hatch_counter                0
has_gender_differences       0
growth_rate_id               0
forms_switchable             0
is_legendary                 0
is_mythical                  0
order                        0
conquest_order             825
dtype: int64

On a bien **1 025 espèces**, une par ligne. Les seuls manquants sont normaux : `evolves_from_species_id` est vide pour les 541 Pokémon qui n'évoluent de rien, `habitat_id` n'existe que pour les vieilles générations et `conquest_order` concerne un spin-off. On n'utilisera ni l'habitat ni `conquest_order`.

In [5]:
# combien de Pokémon par génération ?
species["generation_id"].value_counts().sort_index()

generation_id
1    151
2    100
3    135
4    107
5    156
6     72
7     88
8     96
9    120
Name: count, dtype: int64

Les générations n'ont pas du tout la même taille : de 72 Pokémon (génération 6) à 156 (génération 5). Il faudra s'en souvenir quand on compare les générations entre elles.

In [6]:
# légendaires, fabuleux, bébés : combien, et est-ce qu'ils se chevauchent ?
print(species[["is_legendary", "is_mythical", "is_baby"]].sum())
print("légendaire ET fabuleux :", ((species.is_legendary == 1) & (species.is_mythical == 1)).sum())

is_legendary    71
is_mythical     23
is_baby         19
dtype: int64
légendaire ET fabuleux : 0


71 légendaires, 23 fabuleux et 19 bébés. Un Pokémon n'est jamais légendaire et fabuleux en même temps, ce sont deux catégories séparées.

## 2. Les formes (`pokemon.csv`)

La taille et le poids sont rangés par **forme** et pas par espèce : une espèce peut avoir plusieurs formes (méga-évolutions, formes d'Alola, de Galar…). Il faut regarder ce que ça donne avant de choisir laquelle garder.

In [7]:
forms = pd.read_csv(RAW / "pokeapi/pokemon.csv")
print(forms.shape)
print(forms["is_default"].value_counts())
forms.head()

(1351, 8)
is_default
1    1025
0     326
Name: count, dtype: int64


,id,identifier,species_id,height,weight,base_experience,order,is_default
0,1,bulbasaur,1,7,69,64.0,1.0,1
1,2,ivysaur,2,10,130,142.0,2.0,1
2,3,venusaur,3,20,1000,236.0,3.0,1
3,4,charmander,4,6,85,62.0,5.0,1
4,5,charmeleon,5,11,190,142.0,6.0,1


In [8]:
# est-ce qu'il y a bien exactement une forme par défaut par espèce ?
forms[forms.is_default == 1].groupby("species_id").size().value_counts()

1    1025
Name: count, dtype: int64

In [9]:
# quelques exemples de formes qui ne sont pas par défaut
forms[forms.is_default == 0].sample(10, random_state=0)[["identifier", "species_id", "height", "weight"]]

,identifier,species_id,height,weight
1147,oricorio-pom-pom,741,6,34
1091,altaria-mega,334,15,206
1167,mimikyu-busted,778,2,7
1270,palkia-origin,484,63,6590
1171,magearna-original,801,10,805
1088,swampert-mega,260,19,1020
1294,miraidon-aquatic-mode,1008,28,2400
1040,basculin-blue-striped,550,10,180
1161,minior-orange,774,3,3
1169,mimikyu-totem-busted,778,4,28


Les formes qui ne sont pas par défaut, ce sont les méga-évolutions, les formes régionales, les variantes de couleur (Minior), les formes de combat (Mimikyu)… En gros ce sont des variantes d'un même Pokémon, pas des Pokémon différents. Chaque espèce a **exactement une** forme par défaut.

Du coup on décide de **ne garder que la forme par défaut** pour la taille, le poids, les types et les stats. Les sondages votent pour un Pokémon, pas pour sa méga-évolution, donc c'est la forme « normale » qui a du sens ici, et ça évite d'avoir plusieurs lignes pour un même Pokémon.

In [10]:
# la taille est en décimètres et le poids en hectogrammes : on regarde les extrêmes
default = forms[forms.is_default == 1]
default[["height", "weight"]].describe()

,height,weight
count,1025.000000,1025.000000
mean,12.116098,669.818537
std,12.481673,1212.754122
min,1.000000,1.000000
25%,5.000000,85.000000
50%,10.000000,280.000000
75%,15.000000,700.000000
max,200.000000,9999.000000


In [11]:
default.sort_values("weight", ascending=False).head(5)[["identifier", "height", "weight"]]

,identifier,height,weight
789,cosmoem,1,9999
796,celesteela,92,9999
382,groudon,35,9500
889,eternatus,200,9500
749,mudsdale,25,9200


Attention aux unités : la taille est en décimètres et le poids en hectogrammes (on divisera par 10 pour avoir des mètres et des kilos). Le poids est très asymétrique (médiane 28 kg, mais jusqu'à 999,9 kg). Les 999,9 kg de Cosmovum et Bamboiselle sont les vraies valeurs du jeu (c'est le maximum possible), pas une erreur de saisie.

### Ce qu'on a fait et pourquoi

**1. Regarder les formes avant de choisir.** Avant de joindre la taille et le poids, on a regardé combien de formes il y a par espèce et à quoi ressemblent celles qui ne sont pas « par défaut » (un échantillon au hasard, avec une graine fixe pour retomber sur le même). C'est ce qui nous a montré que ce sont des variantes (méga-évolutions, formes régionales…) et pas des Pokémon différents.

**2. Garder une seule forme par espèce.** On a pris la forme par défaut parce que les sondages votent pour « le Pokémon », pas pour une de ses variantes, et parce qu'on veut **une ligne = un Pokémon** : avec plusieurs formes par espèce, un même Pokémon compterait plusieurs fois dans toutes les analyses. On a vérifié qu'il y a exactement une forme par défaut par espèce, sinon le choix n'aurait pas été possible.

**3. `describe()` et les extrêmes pour les unités.** On a regardé le résumé et les Pokémon les plus lourds pour comprendre les unités (décimètres, hectogrammes) et vérifier que les valeurs bizarres (le maximum à 999,9 kg) sont de vraies valeurs du jeu et pas des erreurs. On regarde la médiane à côté de la moyenne parce que le poids est très asymétrique : quelques Pokémon énormes tirent la moyenne vers le haut.

## 3. Les types (`pokemon_types.csv` + `types.csv`)

Chaque forme a un ou deux types, rangés par `slot` (1 = type principal, 2 = type secondaire).

In [12]:
ptypes = pd.read_csv(RAW / "pokeapi/pokemon_types.csv")
types = pd.read_csv(RAW / "pokeapi/types.csv")
print(types[["id", "identifier"]].to_string(index=False))
ptypes.head()

   id identifier
    1     normal
    2   fighting
    3     flying
    4     poison
    5     ground
    6       rock
    7        bug
    8      ghost
    9      steel
   10       fire
   11      water
   12      grass
   13   electric
   14    psychic
   15        ice
   16     dragon
   17       dark
   18      fairy
   19    stellar
10001    unknown
10002     shadow


,pokemon_id,type_id,slot
0,1,12,1
1,1,4,2
2,2,12,1
3,2,4,2
4,3,12,1


In [13]:
ptypes = ptypes.merge(types[["id", "identifier"]], left_on="type_id", right_on="id")
types_wide = ptypes.pivot(index="pokemon_id", columns="slot", values="identifier")
types_wide.columns = ["type1", "type2"]
types_wide.head()

,type1,type2
pokemon_id,,
1,grass,poison
2,grass,poison
3,grass,poison
4,fire,NaN
5,fire,NaN


Les types sont en anglais. `stellar`, `unknown` et `shadow` sont des types spéciaux qui ne sont portés par aucune forme normale. Un peu moins de la moitié des Pokémon n'ont qu'un seul type (le `type2` vide n'est donc pas un manquant, c'est juste « pas de second type »).

### Ce qu'on a fait et pourquoi

On a fait un **pivot** : le fichier de PokéAPI a une ligne par (forme, type), et on le transforme en une ligne par forme avec deux colonnes `type1` et `type2`. On garde l'ordre du `slot` parce que le type principal n'a pas le même sens que le second (un Pokémon « Plante / Poison » est d'abord un Pokémon Plante). Un `type2` vide n'est pas un manquant à corriger : c'est juste un Pokémon qui n'a qu'un type, donc on ne le remplit pas.

## 4. Les statistiques de combat (`pokemon_stats.csv` + `stats.csv`)

In [14]:
pstats = pd.read_csv(RAW / "pokeapi/pokemon_stats.csv")
stats = pd.read_csv(RAW / "pokeapi/stats.csv")
print(stats[["id", "identifier"]].to_string(index=False))
pstats.groupby("pokemon_id").size().value_counts()

 id      identifier
  1              hp
  2          attack
  3         defense
  4  special-attack
  5 special-defense
  6           speed
  7        accuracy
  8         evasion
  9         special


6    1351
Name: count, dtype: int64

In [15]:
pstats = pstats.merge(stats[["id", "identifier"]], left_on="stat_id", right_on="id")
stats_wide = pstats.pivot(index="pokemon_id", columns="identifier", values="base_stat")
stats_wide = stats_wide[["hp", "attack", "defense", "special-attack", "special-defense", "speed"]]
stats_wide.columns = ["pv", "attaque", "defense", "attaque_spe", "defense_spe", "vitesse"]
stats_wide["total_stats"] = stats_wide.sum(axis=1)
stats_wide.describe().T

,count,mean,std,min,25%,50%,75%,max
pv,1351.0,71.673575,26.827529,1.0,55.0,70.0,85.0,255.0
attaque,1351.0,82.757957,32.351906,5.0,60.0,80.0,102.5,190.0
defense,1351.0,76.193930,30.712902,5.0,55.0,70.0,95.0,250.0
attaque_spe,1351.0,75.418209,34.026272,10.0,50.0,69.0,97.0,216.0
defense_spe,1351.0,74.115470,27.800826,20.0,54.0,70.0,90.0,250.0
vitesse,1351.0,72.000000,30.642043,5.0,49.0,70.0,95.0,200.0
total_stats,1351.0,452.159141,122.456241,175.0,340.0,475.0,530.0,1125.0


Les 6 stats sont présentes pour toutes les formes (`accuracy`, `evasion` et `special` ne sont pas des stats de base). Le total des stats va de 175 à 1 125 (le maximum vient d'une forme spéciale ; on ne garde de toute façon que les formes par défaut).

### Ce qu'on a fait et pourquoi

Même chose que pour les types : un **pivot** pour passer d'une ligne par (forme, stat) à une colonne par stat. On ne garde que les 6 stats de base (les autres, comme `accuracy`, n'en sont pas) et on ajoute le **total des stats**, parce que c'est la mesure habituelle de la « force » d'un Pokémon chez les joueurs, plus simple à comparer qu'un ensemble de 6 chiffres.

## 5. Les noms (`pokemon_species_names.csv`)

On garde l'anglais (clé pour Google Trends et les sondages), le français, et le japonais en kana et en romaji (pour l'axe Noms).

In [16]:
names = pd.read_csv(RAW / "pokeapi/pokemon_species_names.csv")
langs = pd.read_csv(RAW / "pokeapi/languages.csv")
langs[["id", "identifier"]]

,id,identifier
0,1,ja-hrkt
1,2,ja-roma
2,3,ko
3,4,zh-hant
4,5,fr
5,6,de
6,7,es
7,8,it
8,9,en
9,10,cs


In [17]:
keep = {9: "nom_en", 5: "nom_fr", 1: "nom_ja", 2: "nom_ja_roma"}
names_wide = (names[names.local_language_id.isin(keep)]
              .pivot(index="pokemon_species_id", columns="local_language_id", values="name")
              .rename(columns=keep)[list(keep.values())])
names_wide.head()

local_language_id,nom_en,nom_fr,nom_ja,nom_ja_roma
pokemon_species_id,,,,
1,Bulbasaur,Bulbizarre,フシギダネ,Fushigidane
2,Ivysaur,Herbizarre,フシギソウ,Fushigisou
3,Venusaur,Florizarre,フシギバナ,Fushigibana
4,Charmander,Salamèche,ヒトカゲ,Hitokage
5,Charmeleon,Reptincel,リザード,Lizardo


In [18]:
names_wide.isna().sum()

local_language_id
nom_en         0
nom_fr         0
nom_ja         0
nom_ja_roma    0
dtype: int64

Les 4 noms sont présents pour les 1 025 espèces, aucun manquant.

## 6. Le stade d'évolution

Il n'y a pas de colonne « stade » toute faite. On le déduit de `evolves_from_species_id` : un Pokémon qui n'évolue de rien est au stade 1, celui qui évolue d'un stade 1 est au stade 2, etc.

On calcule aussi si le Pokémon est **la dernière évolution** de sa lignée (personne n'évolue de lui).

In [19]:
parent = species.set_index("id")["evolves_from_species_id"]

def stade(i):
    s = 1
    while pd.notna(parent[i]):
        i = int(parent[i])
        s += 1
    return s

species["stade"] = species["id"].map(stade)
a_des_enfants = set(species["evolves_from_species_id"].dropna().astype(int))
species["evolution_finale"] = ~species["id"].isin(a_des_enfants)
species["stade"].value_counts().sort_index()

stade
1    541
2    363
3    121
Name: count, dtype: int64

541 Pokémon au stade 1, 363 au stade 2 et 121 au stade 3.

In [20]:
# le cas des bébés : ils ont été ajoutés après coup devant des lignées existantes
noms = names_wide["nom_fr"]
bebes = species[species.is_baby == 1]
pd.DataFrame({
    "bebe": bebes["id"].map(noms).values,
    "stade_bebe": bebes["stade"].values,
    "evolue_en": [noms[species.loc[species.evolves_from_species_id == i, "id"]].tolist() for i in bebes["id"]],
})

,bebe,stade_bebe,evolue_en
0,Pichu,1,[Pikachu]
1,Mélo,1,[Mélofée]
2,Toudoudou,1,[Rondoudou]
3,Togepi,1,[Togetic]
4,Debugant,1,"[Kicklee, Tygnon, Kapoera]"
5,Lippouti,1,[Lippoutou]
6,Élekid,1,[Élektek]
7,Magby,1,[Magmar]
8,Azurill,1,[Marill]
9,Okéoké,1,[Qulbutoké]


Le problème des **bébés** : ils ont été ajoutés dans les générations 2 et 4 devant des lignées qui existaient déjà. Du coup Pikachu passe au stade 2 parce que Pichu existe, alors que Pikachu a été créé comme un Pokémon de base. Pareil pour Ronflex, Leveinard, Lucario, etc.

On a hésité à mettre les bébés à un « stade 0 » pour que Pikachu reste au stade 1. Finalement **on garde la lignée telle qu'elle est aujourd'hui** (Pichu stade 1, Pikachu stade 2, Raichu stade 3) : c'est comme ça que les joueurs actuels voient la lignée, et ça évite d'inventer une règle à part pour 19 Pokémon.

In [21]:
# les Pokémon qui n'ont aucune évolution (ni avant ni après)
seul = (species["stade"] == 1) & species["evolution_finale"]
print("Pokémon sans aucune évolution :", seul.sum())
print("dont légendaires ou fabuleux :", (seul & ((species.is_legendary == 1) | (species.is_mythical == 1))).sum())

Pokémon sans aucune évolution : 201
dont légendaires ou fabuleux : 84


201 Pokémon n'ont aucune évolution, et parmi eux 84 sont légendaires ou fabuleux. Pour eux le « stade 1 » ne veut pas dire « Pokémon de base faible » : un Mewtwo ou un Lokhlass n'a rien à voir avec un Chenipan.

Du coup on décide de les mettre dans **une catégorie à part, « sans évolution »**. Sinon ils seraient mélangés avec les vrais stades 1 et ils brouilleraient toutes les comparaisons par stade.

On crée donc une colonne `stade_categorie` : « sans évolution » pour ces 201 Pokémon, et « stade 1 », « stade 2 », « stade 3 » pour les autres. On garde quand même la colonne `stade` en nombre, ça peut servir pour des corrélations.

In [22]:
species["stade_categorie"] = "stade " + species["stade"].astype(str)
species.loc[seul, "stade_categorie"] = "sans évolution"
species["stade_categorie"].value_counts()

stade_categorie
stade 2           363
stade 1           340
sans évolution    201
stade 3           121
Name: count, dtype: int64

### Ce qu'on a fait et pourquoi

**1. Calculer le stade en remontant la lignée.** Il n'y a pas de colonne « stade », alors on l'a calculée : pour chaque Pokémon, on remonte de parent en parent (`evolves_from_species_id`) jusqu'à un Pokémon qui n'évolue de rien, et on compte les marches. C'est une petite boucle simple, pas besoin d'algorithme de graphe pour des lignées de 3 Pokémon au plus. La « dernière évolution », c'est un Pokémon dont personne n'évolue.

**2. Garder les bébés tels quels.** On a hésité à créer un « stade 0 » pour les bébés (pour que Pikachu reste au stade 1), mais ça voulait dire inventer une règle à part. On a préféré garder la lignée comme les joueurs la voient aujourd'hui.

**3. Une catégorie « sans évolution ».** On a séparé les Pokémon qui n'ont aucune évolution parce que beaucoup sont des légendaires : les mettre au stade 1 avec les Pokémon de début de lignée aurait mélangé deux choses qui n'ont rien à voir. On garde quand même le stade en nombre à côté, parce qu'une variable numérique sert pour les corrélations, et la variable en catégories pour les comparaisons de groupes.

## 7. Les starters (`starters.csv`)

Fichier construit à la main à partir de Bulbapedia : les 27 lignées de starters, avec leurs évolutions.

In [23]:
starters = pd.read_csv(RAW / "starters/starters.csv")
print(starters.shape)
starters.head(6)

(81, 7)


,dex,nom_en,nom_fr,generation,type_starter,dex_starter,stade
0,1,Bulbasaur,Bulbizarre,1,Plante,1,1
1,2,Ivysaur,Herbizarre,1,Plante,1,2
2,3,Venusaur,Florizarre,1,Plante,1,3
3,4,Charmander,Salamèche,1,Feu,4,1
4,5,Charmeleon,Reptincel,1,Feu,4,2
5,6,Charizard,Dracaufeu,1,Feu,4,3


In [24]:
starters.groupby(['generation', 'type_starter']).size().unstack()

type_starter,Eau,Feu,Plante
generation,,,
1,3,3,3
2,3,3,3
3,3,3,3
4,3,3,3
5,3,3,3
6,3,3,3
7,3,3,3
8,3,3,3
9,3,3,3


On a bien les 27 lignées de starters (3 par génération et par type), soit 81 Pokémon.

## 8. Les sources de popularité qui ont déjà un numéro de Pokédex

Avant de les ajouter, un choix important : **on ne fait pas de score de popularité combiné**. Chaque source garde sa propre colonne. Un Pokémon peut être très aimé chez un public (ou dans un pays) et beaucoup moins ailleurs, et les sources n'ont pas du tout le même nombre de votants (52 000 sur Reddit, plus de 500 000 au Johto Times, des millions de recherches Google). Une moyenne ferait disparaître ces différences, alors que c'est justement intéressant. La façon de comparer les sources entre elles est faite dans le notebook 02.

- **Reddit 2019** : le fichier n'a pas de colonne « numéro », mais l'index ressemble à un numéro de Pokédex. Il faut le vérifier en comparant les noms avec PokéAPI.
- **Johto Times 2025** : il a une colonne `dex`, on vérifie aussi les noms.
- **Pokémon of the Year 2020** : il n'a que des noms, l'harmonisation des noms est faite par Lucas, donc on ne le joint pas ici.

In [25]:
reddit = pd.read_csv(RAW / "reddit_2019/df_ranked.csv", index_col=0)
print(reddit.shape, "index unique :", reddit.index.is_unique, "| de", reddit.index.min(), "à", reddit.index.max())
reddit.head()

(809, 9) index unique : True | de 1 à 809


,name,votes,types,generation,family,sprite_source,generation_color,ranking_overall,ranking_generation
734,Yungoos,0.0,Normal,7,Yungoos,https://raw.githubusercontent.com/PokeAPI/spri...,#E89483,809,88.0
266,Silcoon,0.0,Bug,3,Wurmple,https://raw.githubusercontent.com/PokeAPI/spri...,#9CD7C8,808,135.0
603,Eelektrik,0.0,Electric,5,Tynamo,https://raw.githubusercontent.com/PokeAPI/spri...,#9FCADF,807,155.0
574,Gothita,0.0,Psychic,5,Gothita,https://raw.githubusercontent.com/PokeAPI/spri...,#9FCADF,806,156.0
343,Baltoy,1.0,Ground Psychic,3,Baltoy,https://raw.githubusercontent.com/PokeAPI/spri...,#9CD7C8,805,134.0


In [26]:
# est-ce que l'index est bien le numéro de Pokédex ? on compare les noms
comp = pd.DataFrame({"reddit": reddit["name"], "pokeapi": names_wide.loc[reddit.index, "nom_en"]})
diff = comp[comp.reddit.str.lower() != comp.pokeapi.str.lower()]
print(len(diff), "noms différents")
diff

11 noms différents


,reddit,pokeapi
788,Tapu-Fini,Tapu Fini
787,Tapu-Bulu,Tapu Bulu
786,Tapu-Lele,Tapu Lele
669,Flabebe,Flabébé
29,Nidoran-F,Nidoran♀
439,Mime-Jr,Mime Jr.
772,Type-Null,Type: Null
785,Tapu-Koko,Tapu Koko
32,Nidoran-M,Nidoran♂
122,Mr-Mime,Mr. Mime


L'index de Reddit est bien le numéro de Pokédex : les 11 noms qui diffèrent sont juste écrits autrement (tirets à la place des espaces, ♀/♂ écrits F/M, accents et apostrophes enlevés). Ce sont les mêmes Pokémon.

In [27]:
reddit["votes"].describe()

count     809.000000
mean       65.173053
std       118.625512
min         0.000000
25%         9.000000
50%        23.000000
75%        62.000000
max      1107.000000
Name: votes, dtype: float64

Les votes Reddit sont très concentrés : la moitié des Pokémon ont 23 votes ou moins, alors que le premier en a 1 107. Le total fait 52 725 votes, un peu moins que les 52 836 réponses du sondage (sans doute des réponses qui ne correspondaient à aucun Pokémon).

In [28]:
johto = pd.read_csv(RAW / "johto_times_2025/johto_times_2025_preliminaire.csv")
print(johto.shape, "dex unique :", johto.dex.is_unique)
comp = johto.set_index("dex")[["name_en"]].join(names_wide["nom_en"])
diff = comp[comp.name_en.str.lower() != comp.nom_en.str.lower()]
print(len(diff), "noms différents")
diff

(1025, 10) dex unique : True
0 noms différents


,name_en,nom_en
dex,,


Pour le Johto Times, les numéros, les noms et les générations collent parfaitement avec PokéAPI.

In [29]:
# la génération donnée par le Johto Times correspond-elle à celle de PokéAPI ?
g = johto.set_index("dex")["generation"]
(g != species.set_index("id")["generation_id"]).sum()

np.int64(0)

### Ce qu'on a fait et pourquoi

**1. Pas de score combiné.** On a choisi de garder chaque source de popularité dans sa propre colonne au lieu de faire une moyenne. Les sources n'ont pas le même public ni la même taille, et une moyenne effacerait justement les différences qu'on veut étudier. La comparaison des sources se fait dans le notebook 02, avec des percentiles.

**2. Vérifier les identifiants en comparant les noms.** Reddit n'a pas de colonne « numéro », seulement un index qui y ressemble. Plutôt que de le croire, on a comparé ses noms avec ceux de PokéAPI (en minuscules pour ne pas compter les différences de majuscules) et on a regardé à la main les noms qui diffèrent. Pour le Johto Times, on a fait pareil avec les noms et la génération. C'est le moyen le plus simple de s'assurer qu'on ne colle pas les votes d'un Pokémon sur un autre.

## 9. La jointure

On part des espèces et on ajoute tout le reste par le numéro de Pokédex. Pour la taille, le poids, les types et les stats, on prend **la forme par défaut** de chaque espèce.

In [30]:
colors = pd.read_csv(RAW / "pokeapi/pokemon_colors.csv").set_index("id")["identifier"]
shapes = pd.read_csv(RAW / "pokeapi/pokemon_shapes.csv").set_index("id")["identifier"]

base = species.rename(columns={"id": "dex", "generation_id": "generation"})
table = pd.DataFrame({
    "dex": base["dex"],
    "generation": base["generation"],
    "couleur": base["color_id"].map(colors),
    "forme_corps": base["shape_id"].map(shapes),
    "legendaire": base["is_legendary"].astype(bool),
    "fabuleux": base["is_mythical"].astype(bool),
    "bebe": base["is_baby"].astype(bool),
    "stade": base["stade"],
    "stade_categorie": base["stade_categorie"],
    "evolution_finale": base["evolution_finale"],
    "taux_capture": base["capture_rate"],
}).set_index("dex")

table = names_wide.join(table)

# forme par défaut : on passe de l'id de forme à l'id d'espèce
d = default.set_index("id")
phys = pd.DataFrame({"taille_m": d["height"] / 10, "poids_kg": d["weight"] / 10}, index=d.index)
phys = phys.join(types_wide).join(stats_wide)
phys.index = d.loc[phys.index, "species_id"]
table = table.join(phys)

# starters
st = starters.set_index("dex")[["type_starter", "dex_starter"]]
table = table.join(st)
table["starter"] = table["type_starter"].notna()

# popularité
table["reddit_votes"] = reddit["votes"]
table["johto_pct_generation"] = johto.set_index("dex")["pct_generation"]
table["johto_rang_generation"] = johto.set_index("dex")["rang_generation"]

table.index.name = "dex"
print(table.shape)
table.head()

(1025, 31)


,nom_en,nom_fr,nom_ja,nom_ja_roma,generation,couleur,forme_corps,legendaire,fabuleux,bebe,stade,stade_categorie,evolution_finale,taux_capture,taille_m,poids_kg,type1,type2,pv,attaque,defense,attaque_spe,defense_spe,vitesse,total_stats,type_starter,dex_starter,starter,reddit_votes,johto_pct_generation,johto_rang_generation
dex,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,Bulbasaur,Bulbizarre,フシギダネ,Fushigidane,1,green,quadruped,False,False,False,1,stade 1,False,45,0.7,6.9,grass,poison,45,49,49,65,65,45,318,Plante,1.0,True,710.0,2.93,7
2,Ivysaur,Herbizarre,フシギソウ,Fushigisou,1,green,quadruped,False,False,False,2,stade 2,False,45,1.0,13.0,grass,poison,60,62,63,80,80,60,405,Plante,1.0,True,83.0,0.63,50
3,Venusaur,Florizarre,フシギバナ,Fushigibana,1,green,quadruped,False,False,False,3,stade 3,True,45,2.0,100.0,grass,poison,80,82,83,100,100,80,525,Plante,1.0,True,127.0,1.35,24
4,Charmander,Salamèche,ヒトカゲ,Hitokage,1,red,upright,False,False,False,1,stade 1,False,45,0.6,8.5,fire,NaN,39,52,43,60,50,65,309,Feu,4.0,True,374.0,0.77,37
5,Charmeleon,Reptincel,リザード,Lizardo,1,red,upright,False,False,False,2,stade 2,False,45,1.1,19.0,fire,NaN,58,64,58,80,65,80,405,Feu,4.0,True,70.0,0.25,89


### Ce qu'on a fait et pourquoi

On a fait toutes les jointures **sur le numéro de Pokédex**, en partant des espèces (la table qui a exactement une ligne par Pokémon) et en ajoutant le reste avec des `join`. Partir des espèces garantit qu'on garde bien les 1 025 Pokémon, même ceux qui n'ont pas de valeur dans une source (ils ont juste un manquant). Pour la taille, le poids, les types et les stats, qui sont rangés par forme, on repasse de l'identifiant de forme à l'identifiant d'espèce avant de joindre. On convertit aussi les unités ici (mètres, kilos) pour que la table soit lisible directement.

## 10. Contrôles de la jointure

Juste les contrôles qui disent si la jointure s'est bien passée (les contrôles de qualité complets, c'est la partie de Lucas).

In [31]:
print("lignes :", len(table), "| dex unique :", table.index.is_unique)
table.isna().sum()[lambda s: s > 0]

lignes : 1025 | dex unique : True


type2           499
type_starter    944
dex_starter     944
reddit_votes    216
dtype: int64

In [32]:
# les manquants de Reddit doivent être exactement les générations 8 et 9
table.loc[table.reddit_votes.isna(), "generation"].value_counts().sort_index()

generation
8     96
9    120
Name: count, dtype: int64

La jointure s'est bien passée : 1 025 lignes, un numéro unique par ligne. Les seuls manquants sont attendus : pas de second type (499), pas starter (944), et Reddit qui ne couvre pas les générations 8 et 9 (216 = 96 + 120).

In [33]:
table.dtypes

nom_en                       str
nom_fr                       str
nom_ja                       str
nom_ja_roma                  str
generation                 int64
couleur                      str
forme_corps                  str
legendaire                  bool
fabuleux                    bool
bebe                        bool
stade                      int64
stade_categorie              str
evolution_finale            bool
taux_capture               int64
taille_m                 float64
poids_kg                 float64
type1                        str
type2                        str
pv                         int64
attaque                    int64
defense                    int64
attaque_spe                int64
defense_spe                int64
vitesse                    int64
total_stats                int64
type_starter                 str
dex_starter              float64
starter                     bool
reddit_votes             float64
johto_pct_generation     float64
johto_rang

### Ce qu'on a fait et pourquoi

Les contrôles sont faits pour vérifier que la jointure n'a rien cassé : le nombre de lignes, l'unicité du numéro, et la liste des manquants. Pour chaque manquant, on vérifie qu'il a une explication (pas de second type, pas starter, Reddit qui s'arrête à la génération 7). C'est pour ça qu'on regarde par exemple la génération des Pokémon sans vote Reddit : si on en avait trouvé en génération 3, ça aurait voulu dire qu'un nom ne s'était pas raccroché.

## 11. Export

In [34]:
table.to_csv(OUT / "table_maitre.csv")
print("écrit :", OUT / "table_maitre.csv", table.shape)

écrit : ../data/processed/table_maitre.csv (1025, 31)
